<a href="https://colab.research.google.com/github/sudipatgit/LogicMojo-GenAI-AgenticAI-Sudip123/blob/main/Copy_of_03_python_data_analysis_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Data Analysis Practice · Delivery Operations

An online retailer receives delivery updates from several carriers. A case can appear more than once as new updates arrive. The operations lead wants to know which cases are still active, which delays need attention, and what to include in a daily briefing.

Download [delivery_exceptions.csv](data/delivery_exceptions.csv) and run the setup cell. Run Jupyter from the notebook's folder so the CSV is under `data/`. In Colab, upload the CSV through the **Files** panel and update the path as shown in the setup comment. Use Python 3.11 or newer. If packages are missing, run `%pip install numpy pandas matplotlib seaborn requests` in a notebook cell.

Write your answer in the blank cell under each question. You can reuse your work on later questions. When a question refers to **cleaned cases**, use normalized text, valid delay values, and the latest update per case. An **active** case has status `open` or `in_progress`.

Refer to [Python Tools](01_python_tools.ipynb) and the [NumPy, pandas and chart sections](02_data_and_llms.ipynb) for class examples.

## Load the CSV

| Column | What it means |
| --- | --- |
| `case_id`, `order_id` | Delivery exception and associated order. A case can have multiple updates. |
| `opened_at`, `updated_at` | When the case was opened and when this update was recorded. |
| `region`, `city`, `carrier` | Where the order was going and who was handling it. |
| `issue_type` | Reason for the case, such as `delayed`, `damaged`, or `missing_parcel`. |
| `status`, `priority` | Current state at the time of the update and its assigned priority. |
| `delay_minutes` | Reported minutes past the delivery promise. Zero is valid. Blanks, labels such as `pending sync`, and negative values are unknown for analysis. |
| `customer_message` | Short example of what the customer reported. |

The setup loads the raw export so you can explore it in the questions below.

## A. Explore the CSV

### 1. What came in the export?

Show the first five and last five rows, the shape, and the column names. Which column identifies an update's case? Which column tells you when that update happened?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

csv_path = Path("data/delivery_exceptions.csv")
delivery_updates = pd.read_csv(csv_path)
display(delivery_updates.head())
display(delivery_updates.tail())
print(delivery_updates.shape)
columns = delivery_updates.columns.tolist()
print("columns are: ", columns)
print("column name for update case: ", columns[8])
print("column name to show when update happened: ", columns[3])




### 2. How many cases are there?

Count the raw rows and the distinct case IDs. List the IDs that appear more than once and count how many updates each has. What do the two counts tell you about how the export was produced?

In [ ]:
distinct_case_ids = delivery_updates["case_id"].unique()

duplicate_case_ids = delivery_updates["case_id"].value_counts()
duplicate_case_ids = duplicate_case_ids[duplicate_case_ids > 1]

print("Total number of rows: ", delivery_updates.shape[0])
print("Number of unique case ids: ", distinct_case_ids.size)
print(duplicate_case_ids)


### 3. Where is the data incomplete?

Count missing values in each raw column. Then inspect `delay_minutes` and list examples of missing, invalid, and valid values. Which of those categories do blanks, `pending sync`, negative numbers, and zero belong to?

In [ ]:
columns_with_missing_value = delivery_updates.isna().sum()[lambda x: x > 0]

missing_delay_minutes = delivery_updates['delay_minutes'].isna()
row_indexes_in_missing_delay_minutes = missing_delay_minutes.index[missing_delay_minutes].tolist()
numeric_vals = pd.to_numeric(delivery_updates['delay_minutes'], errors='coerce')
valid_rows = numeric_vals >= 0
invalid_rows = (numeric_vals.isna() & ~missing_delay_minutes) | (numeric_vals < 0)

print("Count of missing values in each column:\n", columns_with_missing_value)
print("\nRow indexes of missing values in delay_minutes: ", missing_delay_minutes.index[missing_delay_minutes].tolist())
print("\nRow indexes and invalid values in delay_minutes: \n")
print(delivery_updates['delay_minutes'][invalid_rows])
print("\nRow indexes and valid values in delay_minutes: \n")
print(delivery_updates['delay_minutes'][valid_rows])

### 4. Which categories need standardization?

Show the distinct raw values of `status`, `region`, `carrier`, and `priority`. Find two labels that would be counted separately in a raw grouping even though they have the same meaning.

In [ ]:
unique_region = delivery_updates['region'].str.strip().str.upper().unique()
unique_carrier = delivery_updates['carrier'].str.strip().str.upper().unique()
unique_priority = delivery_updates['priority'].str.strip().str.upper().unique()

delivery_updates['status_norm'] = (
    delivery_updates['status']
    .str.strip()
    .str.upper()
    .str.replace(r'[\s\-_]+', '', regex=True)
)

delivery_updates['clean_status'] = delivery_updates['status'].str.strip().str.upper()

grouped = delivery_updates.groupby('status_norm')['clean_status'].unique()
inconsistent = grouped[grouped.apply(len) > 1]

print("\nDistinct raw values of status: \n", delivery_updates['clean_status'].unique())
print("\nDistinct raw values of region: \n", unique_region)
print("\nDistinct raw values of carrier: \n", unique_carrier)
print("\nDistinct raw values of priority: \n", unique_priority)
print("\nLabels that would be counted separately in a raw grouping, are: \n", inconsistent)

### 5. What happened to case 7008?

Use `.loc` to display all updates for case 7008 with just the case ID, update time, status, and delay. Sort them by update time. Then use `.iloc` to select the last row of that result. Which status should a lookup tool return?

In [ ]:
case_7008 = delivery_updates.loc[delivery_updates['case_id'] == 7008, ['case_id', 'updated_at', 'status', 'delay_minutes']]
case_7008_sorted = case_7008.sort_values('updated_at')
last_update_7008 = case_7008_sorted.iloc[-1]
print("\ncase 7008: \n", case_7008)
print("\nStatus of last update for case 7008: ", last_update_7008['status'])

## B. Clean and prepare the data

### 6. Can you normalize the text labels?

Create a copy and trim spaces and lowercase the region, carrier, issue type, status, and priority columns. Use a mapping to turn both `in progress` and `in-progress` into `in_progress`. Compare the distinct status labels before and after cleaning.

In [ ]:
copy_delivery_updates = delivery_updates.loc[:, ['region', 'carrier', 'issue_type', 'status', 'priority']]
copy_delivery_updates['region'] = copy_delivery_updates['region'].str.strip().str.lower()
copy_delivery_updates['carrier'] = copy_delivery_updates['carrier'].str.strip().str.lower()
copy_delivery_updates['issue_type'] = copy_delivery_updates['issue_type'].str.strip().str.lower()
copy_delivery_updates['status'] = copy_delivery_updates['status'].str.strip().str.lower()
copy_delivery_updates['priority'] = copy_delivery_updates['priority'].str.strip().str.lower()

status_mapping = {
    'in progress': 'in_progress',
    'in-progress': 'in_progress'
}
copy_delivery_updates['status'] = copy_delivery_updates['status'].replace(status_mapping)

print("\nCleaned values of region, carrier, issue type, status, and priority: \n", copy_delivery_updates[['region', 'carrier', 'issue_type', 'status', 'priority']])
print("\nDistinct status labels before cleaning: \n", delivery_updates['status'].unique())
print("\nDistinct status labels after cleaning: \n", copy_delivery_updates['status'].unique())

### 7. Can you turn a messy delay column into numbers?

Convert `delay_minutes` to numbers with pandas. Treat blanks, unparseable text, and negative values as missing. Keep zero as a valid measurement. How many measurements became missing? Inspect cases 7004, 7005, and 7006 to show the difference.

In [ ]:
missing_count_in_delay_minutes = (numeric_vals.isna()) | (numeric_vals < 0)
case_7004 = delivery_updates.loc[delivery_updates['case_id'] == 7004, ['delay_minutes']]
case_7005 = delivery_updates.loc[delivery_updates['case_id'] == 7005, ['delay_minutes']]
case_7006 = delivery_updates.loc[delivery_updates['case_id'] == 7006, ['delay_minutes']]

print("\nCount of missing values in delay_minutes: ", missing_count_in_delay_minutes.sum())
print("\nCase 7004: ", case_7004)
print("\nCase 7005: ", case_7005)
print("\nCase 7006: ", case_7006)

### 8. How much time passed before each update?

Parse `opened_at` and `updated_at` as datetimes. Create a column giving the minutes between opening and that update. Find the three updates with the largest elapsed time and explain what this measure tells you about a case.

In [ ]:
delivery_updates[['opened_at', 'updated_at']] = delivery_updates[['opened_at', 'updated_at']].apply(pd.to_datetime)
delivery_updates['elapsed_time'] = (delivery_updates['updated_at'] - delivery_updates['opened_at']).dt.total_seconds() / 60

print("\nThree updates with the largest elapsed time: \n", delivery_updates.nlargest(3, 'elapsed_time'))

### 9. Which update should represent each case?

Sort by `updated_at` and keep the latest row for each `case_id`. The export is shuffled, so use the timestamp to decide which row to keep. Compare the status counts before and after deduplication.

In [ ]:
df_new = delivery_updates.sort_values('updated_at')
dedup = df_new.groupby('case_id').tail(1)

print("\nStatus counts before deduplication: \n", delivery_updates['status'].value_counts())
print("\nStatus counts after deduplication: \n", dedup['status'].value_counts())

### 10. Can you make the cleaning reusable?

Package the cleaning and deduplication into `prepare_cases(raw_cases: pd.DataFrame) -> pd.DataFrame`. Return a new table and leave the input intact. Print a short summary of how many rows came in, how many unique cases remained, and how many of those cases had unknown delays.

In [3]:
def prepare_cases(raw_cases: pd.DataFrame, show_prints: bool = True) -> pd.DataFrame:
    df = raw_cases.copy()
    df[['opened_at', 'updated_at']] = df[['opened_at', 'updated_at']].apply(pd.to_datetime)
    df = df.sort_values('updated_at')
    df['delay_minutes'] = pd.to_numeric(df['delay_minutes'], errors='coerce')
    df.loc[df['delay_minutes'] < 0, 'delay_minutes'] = np.nan

    df['region'] = df['region'].str.strip().str.lower()
    df['carrier'] = df['carrier'].str.strip().str.lower()
    df['issue_type'] = df['issue_type'].str.strip().str.lower()
    df['status'] = df['status'].str.strip().str.lower()
    df['priority'] = df['priority'].str.strip().str.lower()

    status_mapping = {
        'in progress': 'in_progress',
        'in-progress': 'in_progress'
    }
    df['status'] = df['status'].replace(status_mapping)

    dedup = df.groupby('case_id').tail(1)

    if show_prints:
        total_rows = raw_cases.shape[0]
        unique_cases = dedup.shape[0]
        unknown_delays = dedup['delay_minutes'].isna().sum()

        print(f"Total rows in input: {total_rows}")
        print(f"Unique cases after deduplication: {unique_cases}")
        print(f"Cases with unknown delays: {unknown_delays}")

    return dedup

## C. Practise Python functions for agent tools

### 11. Can you validate a case ID from a request?

Write `parse_case_id(value: object) -> int`. Accept positive integers and digit strings after trimming spaces. Raise `ValueError` for other values, including booleans. Try it with `7001`, `" 7001 "`, `"ORD-8401"`, and `True`.

In [ ]:
def parse_case_id(value: object) -> int:
    if isinstance(value, bool):
        raise ValueError(f"Invalid case ID: {value}")
    if isinstance(value, int) and value > 0:
        return value
    if isinstance(value, str):
        value = value.strip()
        if value.isdigit():
            return int(value)
    raise ValueError(f"Invalid case ID: {value}")

### 12. Can you look up the latest status?

Convert the cleaned table to a list of dictionaries. Write a lookup function that accepts a case ID and returns a dictionary with the case ID, status, carrier, and issue type. Return an error dictionary if the ID is missing from the dataset, and try a known ID and an absent one.

In [ ]:

class TableToDictList:
    def __init__(self, df: pd.DataFrame):
        self.dict_list = df.to_dict(orient='records')

    def lookup_status(self, case_id: int) -> dict:
        for case in self.dict_list:
            if case['case_id'] == case_id:
                return case
        return {"error": "Case ID not found"}


dict_list_instance = TableToDictList(prepare_cases(delivery_updates))

print(dict_list_instance.lookup_status(7001))
print(dict_list_instance.lookup_status(9999))

### 13. Which cases should the operations team review first?

Write a function that accepts the records, a region, and a delay threshold. Return active cases in that region with a known delay at or above the threshold. Use a lambda to order them by longest delay first, with smaller case IDs first when delays tie. Compare the North region at 120 and 240 minutes.

In [ ]:
def review_cases(records: list, region: str, delay_threshold: float) -> list:
    filtered_cases = [
        case for case in records
        if case['region'] == region.lower() and
           (case['status'] == 'open' or case['status'] == 'in_progress') and
           pd.notna(case.get("delay_minutes")) and
           case['delay_minutes'] >= delay_threshold
    ]
    filtered_cases.sort(key=lambda r: (-r["delay_minutes"], r["case_id"]))
    return filtered_cases

clean_dataframe = prepare_cases(delivery_updates, False)
records_list = clean_dataframe.to_dict(orient='records')
filtered_cases_120 = review_cases(records_list, 'north', 120)
filtered_cases_240 = review_cases(records_list, 'north', 240)

print("\nActive cases in North region with delay >= 120 minutes: \n", filtered_cases_120)
print("\nActive cases in North region with delay >= 240 minutes: \n", filtered_cases_240)

### 14. How can one run avoid reviewing a case twice?

Build a `DeliveryRun` class with a set on each instance. Give it methods to mark an ID as seen, check whether it was seen, and count distinct IDs. Create two instances, mark an ID twice in the first, and show what each instance remembers.

In [ ]:
class DeliveryRun:
    def __init__(self):
        self.seen_ids = set()

    def mark_seen(self, case_id: int):
        self.seen_ids.add(case_id)

    def has_seen(self, case_id: int) -> bool:
        return case_id in self.seen_ids

    def count_distinct(self) -> int:
        return len(self.seen_ids)

instance1 = DeliveryRun()
instance1.mark_seen(7001)
instance1.mark_seen(7001)
instance1.mark_seen(7002)

instance2 = DeliveryRun()
instance2.mark_seen(7005)

print(f"Distinct IDs in Instance 1: {instance1.count_distinct()}")
print(f"Instance 1 has seen 7001?: {instance1.has_seen(7001)}")
print(f"Instance 1 has seen 7003?: {instance1.has_seen(7003)}")

print(f"Distinct IDs in Instance 2: {instance2.count_distinct()}")
print(f"Instance 2 has seen 7001?: {instance2.has_seen(7001)}")

### 15. Can you turn case records into short descriptions?

Write a named function that accepts one case and returns a line such as `Case 7008 | north | resolved`. Apply it to a list of records using Python's built-in `map()`. Use `functools.reduce()` to total the known delays for active cases and compare the result with `sum()`.

In [ ]:
from functools import reduce

def format_case(case: dict) -> str:
    return f"Case {case['case_id']} | {case['region']} | {case['status']}"

formatted_lines = list(map(format_case, records_list))

print("\n--- Formatted Cases ---")
for line in formatted_lines:
    print(line)

# Use functools.reduce() to find total known delays for active cases
def add_active_delay(total: int, case: dict) -> int:
    is_active = case.get("status") == "in_progress" or case.get("status") == "open"
    delay = case.get("delay_minutes")

    if is_active and pd.notna(delay):
        return total + delay
    return total

total_reduce_delay = reduce(add_active_delay, records_list, 0)

total_delay_sum = sum(
    case["delay_minutes"]
    for case in records_list
    if (case.get("status") == "in_progress" or case.get("status") == "open") and pd.notna(case.get("delay_minutes"))
)

print(f"\nTotal Delay via reduce(): {total_reduce_delay}")
print(f"\nTotal Delay via sum():    {total_delay_sum}")
print(f"\nResults match:      {total_reduce_delay == total_delay_sum}")

## D. Analyze the backlog with NumPy and pandas

### 16. How many delays meet each threshold?

Build a NumPy array of known delays from active cases. Use boolean masks to count the delays at or above 60, 120, and 240 minutes. Also report the minimum, median, and maximum. How many active cases were excluded because their delays were unknown?

In [ ]:
clean_df1 = prepare_cases(delivery_updates, False)

active_cases = clean_df1[(clean_df1['status'] == 'open') | (clean_df1['status'] == 'in_progress')]

excluded_active_count = int(active_cases["delay_minutes"].isna().sum())

delays_arr = active_cases["delay_minutes"].dropna().to_numpy(dtype=float)

count_60 = int((delays_arr >= 60).sum())
count_120 = int((delays_arr >= 120).sum())
count_240 = int((delays_arr >= 240).sum())

if delays_arr.size > 0:
    min_delay = float(np.min(delays_arr))
    median_delay = float(np.median(delays_arr))
    max_delay = float(np.max(delays_arr))
else:
    min_delay = median_delay = max_delay = None

print(f"\nCount of Delays >= 60 mins:  {count_60}")
print(f"\nCount of Delays >= 120 mins: {count_120}")
print(f"\nCount of Delays >= 240 mins: {count_240}")
print(f"\nMinimum delay:      {min_delay}")
print(f"\nMedian delay:       {median_delay}")
print(f"\nMaximum delay:      {max_delay}")
print(f"\nExcluded active cases due to unknown delay: {excluded_active_count}")

### 17. What happens when you combine two conditions?

Use `.loc` to find high or urgent active cases with a known delay of at least 120 minutes. Show only the case ID, region, carrier, priority, and delay. Sort the result and use `.iloc` to display the first three matching rows.

In [ ]:
high_or_urgent_active_cases = clean_df1.loc[
    ((clean_df1['priority'] == 'high') | (clean_df1['priority'] == 'urgent')) &
    (clean_df1['status'].isin(['open', 'in_progress'])) &
    (clean_df1['delay_minutes'].notnull()) &
    (clean_df1['delay_minutes'] >= 120),
    ['case_id', 'region', 'carrier', 'priority', 'delay_minutes']
]

matching_rows_sorted = high_or_urgent_active_cases.sort_values(['delay_minutes', 'case_id'], ascending=[True, True])

print("\nHigh or urgent active cases with known delay >= 120 minutes: \n", high_or_urgent_active_cases)
print("\nFirst three matching rows after sorting: \n", matching_rows_sorted.iloc[:3])

### 18. Which carriers have the most active cases?

Group active cases by carrier. For each carrier, show the number of active cases, the number with a known delay, and the average known delay. Why might `.size()` and `.count()` give different numbers?

In [ ]:
grouped_by_carrier = active_cases.groupby('carrier').agg(
    active_cases_count=('delay_minutes', 'size'),
    known_delays_count=('delay_minutes', 'count'),
    avg_delay=('delay_minutes', 'mean')
)

print("\nActive cases grouped by carrier: \n", grouped_by_carrier)
print("\nNote: .size() counts all rows in the group, while .count() counts only non-missing values.")

### 19. Can you assign a label to each known delay?

Use `Series.map()` with a dictionary to give each priority a readable label. Then use `Series.apply()` with a named function to label delays as `unknown`, `under 120 minutes`, or `120 minutes or more`. Show the counts for each delay label among active cases.

In [ ]:
priority_labels = {
    "high": "High Priority",
    "low": "Low Priority",
    "normal": "Normal Priority",
    "urgent": "Urgent Priority",
}

clean_df1["priority_label"] = clean_df1["priority"].map(priority_labels).fillna("Unassigned")

def categorize_delay(delay_val):
    if pd.isna(delay_val):
        return "unknown"
    elif delay_val < 120:
        return "under 120 minutes"
    else:
        return "120 minutes or more"

clean_df1["delay_label"] = clean_df1["delay_minutes"].apply(categorize_delay)

active_delay_counts = clean_df1.loc[clean_df1["status"].isin(["in_progress", "open"]), "delay_label"].value_counts()

print("\n--- DataFrame with new Labeled Columns ---")
print(clean_df1[["priority_label", "delay_label"]])

print("\n--- Count of each delay Label (Active Cases Only) ---")
print(active_delay_counts)

### 20. Which region and carrier combinations need attention?

Build a grouped table by region and carrier for active cases. Show the active count and the number at or above 120 minutes. Calculate the percentage meeting the threshold **among known delays** and use a missing result when a group has no known delays. Which combinations look different when you compare counts with percentages?

In [ ]:
active_df = clean_df1[clean_df1['status'].isin(['open', 'in_progress'])]

active_df["is_120_or_more"] = active_df["delay_minutes"] >= 120

grouped_summary = (
    active_df.groupby(["region", "carrier"], as_index=False)
    .agg(
        active_count=("status", "size"),
        known_delays=("delay_minutes", "count"),
        count_ge_120=("is_120_or_more", "sum"),
    )
)

grouped_summary["pct_ge_120_known"] = (
    grouped_summary["count_ge_120"] / grouped_summary["known_delays"]
) * 100

print(grouped_summary)

## E. Choose and explain a chart

### 21. Which carrier has the most active cases?

Make a Seaborn count plot using active cases by carrier. Add labels above the bars and compare them with your grouped counts. Give the chart a title and label both axes.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns

grouped_counts = (
    active_df.groupby("carrier", as_index=False)
    .agg(
        active_total_rows=("status", "size"),
        known_delay_rows=("delay_minutes", "count"),
    )
    .sort_values(by="active_total_rows", ascending=False)
)

print("\n Active Case Counts Grouped by Carrier: ")
print(grouped_counts.to_string(index=False))

plt.figure(figsize=(6, 5))
ax = sns.countplot(
    data=active_df,
    x="carrier",
    order=grouped_counts["carrier"],
    color="steelblue",
)

ax.bar_label(ax.containers[0])
ax.margins(y=0.10)

plt.title("Active Cases by Carrier")
plt.xlabel("Carrier")
plt.ylabel("Number of Active Cases")
plt.tight_layout()
plt.show()

### 22. How are known delays distributed?

Plot a histogram of known delays for active cases. Draw a line at 120 minutes, then compare the chart with the exact count from a mask. How could changing the histogram bins change what you notice?

In [ ]:
known_delays_with_active_case = active_df['delay_minutes'].dropna()
mask_ge_120 = known_delays_with_active_case >= 120
exact_count_ge_120 = int(mask_ge_120.sum())
total_known = len(known_delays_with_active_case)

print(f"Total active cases with known delay: {total_known}")
print(f"Exact count from >= 120 mins mask:   {exact_count_ge_120}")
print(f"Max known delay: {known_delays_with_active_case.max()} ")

plt.figure(figsize=(7, 3.5))
sns.histplot(
    x=known_delays_with_active_case,
    binwidth=30,
    binrange=(0, known_delays_with_active_case.max()),
    color="seagreen"
)
plt.axvline(
    120,
    color="darkorange",
    linestyle="--",
    # linewidth=2,
    label="120-minute threshold",
)
plt.title("Distribution of Known Delays for Active Cases")
plt.xlabel("Delay (minutes)")
plt.ylabel("Number of Active Cases")
plt.xticks(range(0, 510, 60))
plt.legend()
plt.tight_layout()
plt.show()

### 23. Which regions have a wider spread of delays?

Make a Seaborn box plot of known active delays by region. Compare medians and spread. Choose one region with a point beyond a whisker and look up the case or cases that may explain it.

In [ ]:
known_active_delay = clean_df1[
    clean_df1["status"].isin(["open", "in_progress"]) & clean_df1["delay_minutes"].notna()
]

region_stats = (
    known_active_delay.groupby("region")["delay_minutes"]
    .agg(
        count="count",
        median="median",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
        min_delay="min",
        max_delay="max",
    )
    .reset_index()
)

region_stats["iqr"] = region_stats["q3"] - region_stats["q1"]
region_stats["upper_limit"] = region_stats["q3"] + 1.5 * region_stats["iqr"]

region_order = region_stats.sort_values("median")["region"]

target_region = "east"
target_row = region_stats[region_stats["region"] == target_region].iloc[0]
lower_limit = target_row["q1"] - 1.5 * target_row["iqr"]
upper_limit = target_row["upper_limit"]

outlier_cases = known_active_delay[
    (known_active_delay["region"] == target_region)
    & ((known_active_delay["delay_minutes"] < lower_limit) | (known_active_delay["delay_minutes"] > upper_limit))
]

print("--- Region Wise Delay Summary (Medians & Spread) ---")
print(region_stats.to_string(index=False))

print(f"\n--- Outlier Cases Beyond Whiskers in '{target_region}' (Upper Limit: {upper_limit:.2f} mins) ---")
print(outlier_cases[["case_id", "region", "carrier", "priority", "status", "delay_minutes"]].to_string(index=False))

plt.figure(figsize=(7, 3.5))
sns.boxplot(
    data=known_active_delay,
    x="region",
    y="delay_minutes",
    order=region_order,
    color="lightblue"
)
plt.title("Known Active Delays by Region", pad=12)
plt.xlabel("Region", labelpad=15)
plt.ylabel("Delay (Minutes)", labelpad=15)
plt.tight_layout()
plt.show()

### 24. How did new exceptions change over time?

Count unique cases by the calendar day on which they opened. Plot the counts in date order as a line chart. Identify the busiest day. Explain why cases opened on a day and cases still active on that day are different measures.

In [ ]:
clean_df1["open_date"] = clean_df1["opened_at"].dt.floor("D")
daily_opened = (
    clean_df1.dropna(subset=["open_date"])
    .groupby("open_date", as_index=False)
    .agg(unique_opened_cases=("case_id", "nunique"))
    .sort_values("open_date")
)

max_opened = daily_opened["unique_opened_cases"].max()
busiest_days = daily_opened[daily_opened["unique_opened_cases"] == max_opened]

print("\n--- Busiest Day(s) ---")
for _, row in busiest_days.iterrows():
    print(f"{row['open_date'].strftime('%Y-%m-%d')}: {row['unique_opened_cases']} unique cases")

plt.figure(figsize=(7, 3.5))

sns.lineplot(
    data=daily_opened,
    x="open_date",
    y="unique_opened_cases",
    marker="o",
    linewidth=2.2,
    markersize=7,
    color="steelblue",
)

# Annotate each point with its exact count
for _, row in daily_opened.iterrows():
    plt.annotate(
        f"{int(row['unique_opened_cases'])}",
        (row["open_date"], row["unique_opened_cases"]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontweight="bold",
        fontsize=10,
    )

plt.scatter(
    busiest_days["open_date"],
    busiest_days["unique_opened_cases"],
    color="red",
    s=90,
    zorder=5,
    label=f"Busiest Day ({max_opened} cases)",
)

plt.gca().xaxis.set_major_locator(mdates.DayLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=30, ha="right")

plt.margins(y=0.22)
plt.title("Unique Opened Cases by Calendar Day", fontsize=13, fontweight="bold", pad=10)
plt.xlabel("Calendar Day Opened", fontsize=11, labelpad=6)
plt.ylabel("Unique Cases Opened", fontsize=11, labelpad=6)
plt.legend(frameon=True)

plt.tight_layout()
plt.show()

### 25. Do longer recorded delays come with longer update times?

On the cleaned table, make a scatter plot of delay minutes against minutes from opening to the latest update. Use region as the color and exclude unknown delays. Describe one pattern or exception, and name another factor you would want to check before drawing a conclusion.

In [ ]:
clean_dataframe["mins_open_to_update"] = (
    (clean_dataframe["updated_at"] - clean_dataframe["opened_at"]).dt.total_seconds() / 60
)

plot_df = clean_dataframe.dropna(subset=["delay_minutes", "mins_open_to_update"]).copy()
print("\n Pattern: \n")
print("Dense Cluster Under 500 Minutes Elapsed & 250 Minutes Delay: \nThe vast majority of cases across all four regions - north, south, east, west, are in the bottom-left quadrant")
print("\n Plot also shows wide vertical scatter — meaning how long a ticket remained open (mins_open_to_update) does not strongly predict its delay_minutes")
print("\n Exceptions: Five cases — 2 in North, 2 in East, and 1 in South show a massive 8-hour (480-minute) delay, even though their support tickets were only open for about 1 to 7 hours")

plt.figure(figsize=(7, 3.5))

sns.scatterplot(
    data=plot_df,
    x="mins_open_to_update",
    y="delay_minutes",
    hue="region",
    palette="Set1",
    s=85,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.5,
)

plt.title("Delay Minutes vs. Minutes from Opening to Latest Update by Region", pad=12)
plt.xlabel("Minutes from Opening to Latest Update", labelpad=12)
plt.ylabel("Delay (Minutes)", labelpad=12)
plt.legend(
    title="Region",
    frameon=True,
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
)

plt.tight_layout()
plt.show()

## F. Build Python tools and reports

### 26. Can your program exchange a tool request as JSON?

Start with `{"tool": "get_case_status", "case_id": "7008"}` as JSON text. Parse it, call the lookup tool, and convert the returned dictionary back to JSON text. Decide what error the application should return for malformed JSON, an unsupported tool, and an absent case. Demonstrate one request for each outcome.

In [ ]:
import json

case_records = clean_dataframe.to_dict(orient='records')

def get_case_status(case_id: str) -> dict:

    normalized_id = str(case_id).strip() if case_id is not None else ""

    if not normalized_id:
        return {
            "ok": False,
            "error": "MISSING_CASE_ID",
            "message": "The 'case_id' field is required.",
        }
    for case in case_records:
        if case['case_id'] == int(normalized_id):
            return {
                "ok": True,
                "tool": "get_case_status",
                "result": case['status']
            }
    return {
        "ok": False,
        "error": "CASE_NOT_FOUND",
        "message": f"No case found with case_id '{normalized_id}'.",
    }

SUPPORTED_TOOLS = {
    "get_case_status": get_case_status,
}

def handle_tool_request(raw_json_text: str) -> str:

    try:
        payload = json.loads(raw_json_text)
    except json.JSONDecodeError as exc:
        return json.dumps({
            "ok": False,
            "error": "INVALID_JSON",
            "message": f"Malformed JSON input: {exc.msg} (line {exc.lineno}, column {exc.colno}).",
        })

    if not isinstance(payload, dict):
        return json.dumps({
            "ok": False,
            "error": "INVALID_PAYLOAD",
            "message": "JSON payload must be a JSON object.",
        })

    tool_name = payload.get("tool")
    if tool_name not in SUPPORTED_TOOLS:
        return json.dumps({
            "ok": False,
            "error": "UNSUPPORTED_TOOL",
            "message": f"Unsupported tool '{tool_name}'. Allowed tools: {list(SUPPORTED_TOOLS.keys())}.",
        })

    tool_fn = SUPPORTED_TOOLS[tool_name]
    result_dict = {}
    if tool_name == "get_case_status":
        result_dict = tool_fn(case_id=payload.get("case_id"))
    elif tool_name == "find_delayed_cases":
        result_dict = tool_fn(region=payload.get("region"), min_delay=payload.get("min_delay"))
    return json.dumps(result_dict)

print("\n------- Valid Request --------\n")
req = {
    "tool": "get_case_status",
    "case_id": "7008"
}

json_output = handle_tool_request(json.dumps(req))

print(f"Output JSON: {json_output}")

print("\n------- Request With Unsupported Tool --------\n")
req = {
    "tool": "delete_case",
    "case_id": "7008"
}

json_output = handle_tool_request(json.dumps(req))

print(f"Output JSON: {json_output}")

print("\n------- Request With Absent Case --------\n")
req = {
    "tool": "get_case_status",
    "case_id": "11000"
}

json_output = handle_tool_request(json.dumps(req))

print(f"Output JSON: {json_output}")

print("\n------- Request With Malformed JSON --------\n")

json_output = handle_tool_request('{"tool": "get_case_status", "case_id": "7008"')

print(f"Output JSON: {json_output}")


### 27. Can you wrap an HTTP request in a Python function?

Use the class practice endpoint at `https://jsonplaceholder.typicode.com/todos/1`. Write a function that accepts a task ID, sends a GET request with a timeout, checks the response status, and returns a dictionary with the ID, title, and completion state. Decide how to return a useful error if the request cannot complete. You can use a sample response dictionary when practising offline.

In [ ]:
import requests

def get_task_status(task_id: int) -> dict:
    url = ""
    if isinstance(task_id, int):
        url = f"https://jsonplaceholder.typicode.com/todos/{task_id}"
    elif isinstance(task_id, str):
            if task_id.isdigit():
                task_id = int(task_id)
                url = f"https://jsonplaceholder.typicode.com/todos/{task_id}"
    if not url or int(task_id) < 0:
        return {
            "ok": False,
            "error": "INVALID_TASK_ID",
            "message": "Task ID must be a positive integer.",
        }

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        task_data = response.json()
        return {
            "ok": True,
            "task_id": task_data["id"],
            "title": task_data["title"],
            "completed": task_data["completed"],
        }

    except requests.exceptions.RequestException as e:
        return {
            "ok": False,
            "error": "REQUEST_FAILED",
            "message": f"Request to {url} failed: {e}",
        }

resp = get_task_status(3)
print(f"Response: {resp}")

### 28. Can you save and reload a daily report?

Build a dictionary with the total active cases, unknown delays, counts by carrier, and number of active cases delayed at least 120 minutes. Save it as `practice_delivery_report.json` using UTF-8. Reload it and show that the values match. Convert any pandas or NumPy values to JSON-compatible Python types where needed.

In [ ]:
import json

clean_sorted_df = prepare_cases(delivery_updates, False)
def build_dictionary(data_frame: pd.DataFrame) -> dict:
    active_cases = data_frame['status'].isin(['open', 'in_progress'])
    unknown_delays = data_frame['delay_minutes'].isna().sum()
    carrier_count = data_frame["carrier"].value_counts().to_dict()

    delayed_active_cases = data_frame.loc[
        active_cases & (data_frame['delay_minutes'].notna()) & (data_frame['delay_minutes'] >= 120), 'status'].to_list()

    return {
        "total_active_cases": int(active_cases.sum()),
        "unknown_delays": int(unknown_delays),
        "carrier_counts": carrier_count,
        "active_cases_delayed_by_120": len(delayed_active_cases)
    }

ret_rec = build_dictionary(clean_sorted_df)

print("\n returned record: \n", ret_rec)

delivery_report_path = Path("practice_delivery_report.json")

with delivery_report_path.open("w", encoding="utf-8") as file:
    json.dump(ret_rec, file, indent=2)

with delivery_report_path.open("r", encoding="utf-8") as file:
    saved_report = json.load(file)

print(f"\n Values of Saved report after loading json file: \n {saved_report}")

for key in saved_report:
    val1 = saved_report[key]
    val2 = ret_rec[key]

    print(f"\nkey: {key}")
    print(f"Value before saving to json file: {val2}")
    print(f"Value after saving and loading from json file: {val1}")
    if val1 == val2:
        print("Both values are matched")
    else:
        print("Values are not matched")

### 29. Can your tool validate a delay search request?

Start with `{"tool": "find_delayed_cases", "region": " North ", "min_delay": 120}` as JSON text. Normalize the region and check that it exists in the cleaned table. Accept a non-negative whole number for the threshold. Then reuse your delay search function and return the matching case IDs and count in a dictionary. Return a clear error for an unknown region or invalid threshold.

In [ ]:

def find_delayed_cases(region: str, min_delay: int) -> dict:

    if not isinstance(region, str) or not region.strip():
        return {
            "ok": False,
            "error": "MISSING_REGION",
            "message": "Region must be a non-empty string.",
        }

    if (isinstance(min_delay, bool) or not isinstance(min_delay, int) or min_delay < 0):
        return {
            "ok": False,
            "error": "INVALID_DELAY",
            "message": f"Invalid delay '{min_delay}'. Must be a non-negative integer.",
        }

    normalized_region = region.strip().lower()
    known_region = ["north", "south", "east", "west"]
    if not normalized_region in known_region:
        return {
            "ok": False,
            "error": "INVALID_REGION",
            "message": f"Invalid region '{normalized_region}'.",
        }
    mask = (
        (clean_dataframe["region"] == normalized_region)
        & clean_dataframe["delay_minutes"].notna()
        & (clean_dataframe["delay_minutes"] >= min_delay)
    )

    matching_ids = clean_dataframe.loc[mask, "case_id"].tolist()
    return {
        "ok": True,
        "region": normalized_region,
        "min_delay": min_delay,
        "count": len(matching_ids),
        "case_ids": matching_ids,
    }

SUPPORTED_TOOLS = {
    "get_case_status": get_case_status,
    "find_delayed_cases": find_delayed_cases
}

print("\n------- Valid Request --------\n")
req = {"tool": "find_delayed_cases", "region": " North ", "min_delay": 120}

json_output = handle_tool_request(json.dumps(req))

print(f"Output JSON: {json_output}")

req = {"tool": "find_delayed_cases", "region": "North-East", "min_delay": 120}

json_output = handle_tool_request(json.dumps(req))

print(f"\n Output JSON: {json_output}")

req = {"tool": "find_delayed_cases", "region": "North", "min_delay": -120}

json_output = handle_tool_request(json.dumps(req))

print(f"\n Output JSON: {json_output}")

### 30. Can you build a daily delivery report from a CSV?

Write a function that takes a CSV path and a delay threshold. Load and clean the data, then return a dictionary with the number of active cases, how many have unknown delays, counts by carrier, and how many known delays meet the threshold. Use an f-string to write a one-sentence briefing from these values. Handle a missing file with a clear message. Run the report for 120 and 240 minutes and explain what changed.

In [ ]:
def load_dataframe_from_csv(path: str, threshold_delay: int) -> dict:
    try:
        csv_path = Path(path)
    except (TypeError, ValueError) as exc:
        return {"error": f"Invalid file path provided: {exc}"}
    try:
       delivery_recs = pd.read_csv(csv_path)
    except (TypeError, ValueError) as exc:
        return {"error": f"Invalid file path provided: {exc}"}
    except FileNotFoundError as ex:
        return {"error": f"File not found: {ex}"}
    except IsADirectoryError as exce:
        return {"error": f"It is a directory not actual file: {exce}"}
    except PermissionError as e:
        return {"error": f"File permission error: {e}"}

    delivery_recs['delay_minutes'] = pd.to_numeric(delivery_recs['delay_minutes'], errors='coerce')
    delivery_recs.loc[delivery_recs['delay_minutes'] < 0, 'delay_minutes'] = np.nan

    delivery_recs['region'] = delivery_recs['region'].str.strip().str.lower()
    delivery_recs['carrier'] = delivery_recs['carrier'].str.strip().str.lower()
    delivery_recs['issue_type'] = delivery_recs['issue_type'].str.strip().str.lower()
    delivery_recs['status'] = delivery_recs['status'].str.strip().str.lower()
    delivery_recs['priority'] = delivery_recs['priority'].str.strip().str.lower()

    status_mapping = {
        'in progress': 'in_progress',
        'in-progress': 'in_progress'
    }
    delivery_recs['status'] = delivery_recs['status'].replace(status_mapping)
    delivery_recs = delivery_recs.sort_values('case_id')

    active_cases = delivery_recs[delivery_recs["status"].isin(["open", "in_progress"])]
    unknown_delay_count = int(active_cases["delay_minutes"].isna().sum())
    known_delays = active_cases["delay_minutes"].dropna()
    carrier_count = active_cases["carrier"].value_counts().to_dict()

    result = {
        "provided_threshold_value": threshold_delay,
        "active_cases": len(active_cases),
        "unknown_delays": unknown_delay_count,
        "carrier_count": carrier_count,
        "delays_ge_threshold": int((known_delays >= threshold_delay).sum()),
    }
    briefing = (
        f"There are {result['active_cases']} active cases ({unknown_delay_count} with unknown delays), "
        f"distributed across carriers as {carrier_count}, with {result['delays_ge_threshold']} known "
        f"delays meeting or exceeding the {threshold_delay}-minute threshold."
    )

    result['briefing'] = briefing

    return result

for threshold in [129, 240]:
    ret_val = load_dataframe_from_csv("data/delivery_exceptions.csv", threshold)
    print(f"\n delivery report with threshold value {threshold} mins: \n {ret_val}")

print(f"\n Difference between two reports: \n Difference in count of delays meeting or exceeding the threshold value ")